In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')
 
df = pd.read_csv(r'C:\git\machine learning\house-price-prediction\data\raw\AmesHousing.csv')
print(f"Original shape: {df.shape}")


Original shape: (2930, 82)


In [2]:
 #── CELL 2 — Drop Useless Columns ───────────────────────────────
cols_to_drop = [
    'Order', 'PID',           # ID columns — not features
    'Utilities', 'Street',    # 99%+ same value
    'Condition 2', 'Roof Matl',
    'Low Qual Fin SF', '3Ssn Porch',
    'Pool Area', 'Pool QC',   # 99.6% no pool
    'Misc Feature', 'Misc Val'
]
 
df.drop(columns=cols_to_drop, inplace=True)
print(f"After dropping useless columns: {df.shape}")
 

After dropping useless columns: (2930, 70)


In [3]:
# ── CELL 3 — Remove Outliers ─────────────────────────────────────
# Two houses: huge area but suspiciously low price (found in EDA)
before = len(df)
df = df[~((df['Gr Liv Area'] > 4000) & (df['SalePrice'] < 300000))]
print(f"Removed {before - len(df)} outlier(s). Rows now: {len(df)}")
 

Removed 3 outlier(s). Rows now: 2927


In [4]:
# ── CELL 4 — Handle Missing Values ──────────────────────────────
 
# --- 4a. Structural absence → fill with 'None' ---
# These columns are NaN because the house simply doesn't have that feature
none_cols = [
    'Pool QC', 'Misc Feature', 'Alley', 'Fence', 'Fireplace Qu',
    'Garage Type', 'Garage Finish', 'Garage Qual', 'Garage Cond',
    'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure',
    'BsmtFin Type 1', 'BsmtFin Type 2', 'Mas Vnr Type'
]
# only fill columns that still exist (some were dropped above)
none_cols = [c for c in none_cols if c in df.columns]
for col in none_cols:
    df[col] = df[col].fillna('None')
 
# --- 4b. Numerical structural absence → fill with 0 ---
zero_cols = ['Mas Vnr Area', 'BsmtFin SF 1', 'BsmtFin SF 2',
             'Bsmt Unf SF', 'Total Bsmt SF', 'Bsmt Full Bath', 'Bsmt Half Bath']
zero_cols = [c for c in zero_cols if c in df.columns]
for col in zero_cols:
    df[col] = df[col].fillna(0)
 
# --- 4c. Genuinely missing → impute ---
# Lot Frontage: impute with median of same neighborhood
df['Lot Frontage'] = df.groupby('Neighborhood')['Lot Frontage'] \
                       .transform(lambda x: x.fillna(x.median()))
 
# Garage Yr Blt: no garage → use year house was built
df['Garage Yr Blt'] = df['Garage Yr Blt'].fillna(df['Year Built'])
 
# Fix the data entry error (2207 → 2007)
df['Garage Yr Blt'] = df['Garage Yr Blt'].replace(2207, 2007)
 
# Electrical: only 1 missing → fill with mode
df['Electrical'] = df['Electrical'].fillna(df['Electrical'].mode()[0])
 
print(f"Missing values remaining: {df.isnull().sum().sum()}")

Missing values remaining: 5


In [5]:
# ── CELL 5 — Ordinal Encoding ────────────────────────────────────
# These features have a natural order (Poor → Excellent)
# We map them to numbers so the model understands the ranking
 
qual_map = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
 
qual_cols = [
    'Exter Qual', 'Exter Cond', 'Bsmt Qual', 'Bsmt Cond',
    'Heating QC', 'Kitchen Qual', 'Fireplace Qu',
    'Garage Qual', 'Garage Cond'
]
qual_cols = [c for c in qual_cols if c in df.columns]
for col in qual_cols:
    df[col] = df[col].map(qual_map)
 
# Basement Exposure
df['Bsmt Exposure'] = df['Bsmt Exposure'].map({'None':0,'No':1,'Mn':2,'Av':3,'Gd':4})
 
# Basement Finish Type
bsmt_fin_map = {'None':0,'Unf':1,'LwQ':2,'Rec':3,'BLQ':4,'ALQ':5,'GLQ':6}
df['BsmtFin Type 1'] = df['BsmtFin Type 1'].map(bsmt_fin_map)
df['BsmtFin Type 2'] = df['BsmtFin Type 2'].map(bsmt_fin_map)
 
# Garage Finish
df['Garage Finish'] = df['Garage Finish'].map({'None':0,'Unf':1,'RFn':2,'Fin':3})
 
# Functional (home functionality)
df['Functional'] = df['Functional'].fillna('Typ')
df['Functional'] = df['Functional'].map(
    {'Sal':1,'Sev':2,'Maj2':3,'Maj1':4,'Mod':5,'Min2':6,'Min1':7,'Typ':8}
)
 
# Fence
df['Fence'] = df['Fence'].map({'None':0,'MnWw':1,'GdWo':2,'MnPrv':3,'GdPrv':4})
 
# Paved Drive
df['Paved Drive'] = df['Paved Drive'].map({'N':0,'P':1,'Y':2})
 
# Land Slope
df['Land Slope'] = df['Land Slope'].map({'Gtl':1,'Mod':2,'Sev':3})
 
print("Ordinal encoding done ✓")
 

Ordinal encoding done ✓


In [6]:
# ── CELL 6 — Log Transform Skewed Features ───────────────────────
# From EDA: these were highly skewed (skew > 0.75)
# log1p(x) = log(x+1) — safe for zero values
 
# Target variable
df['SalePrice'] = np.log1p(df['SalePrice'])
 
# Skewed numerical features
skewed_features = [
    'Lot Area', 'Gr Liv Area', 'Total Bsmt SF',
    '1st Flr SF', 'Mas Vnr Area', 'Open Porch SF',
    'Wood Deck SF', 'Lot Frontage'
]
skewed_features = [c for c in skewed_features if c in df.columns]
for col in skewed_features:
    df[col] = np.log1p(df[col])
 
print("Log transform done ✓")
 

Log transform done ✓


In [7]:
#── CELL 7 — One-Hot Encode Nominal Categoricals ────────────────
# Nominal = no natural order (Neighborhood, Roof Style, etc.)
# pd.get_dummies creates one binary column per category
 
before_cols = df.shape[1]
df = pd.get_dummies(df, drop_first=True)
print(f"Columns before encoding : {before_cols}")
print(f"Columns after encoding  : {df.shape[1]}")

Columns before encoding : 70
Columns after encoding  : 189


In [8]:
#── CELL 8 — Feature Scaling ─────────────────────────────────────
# StandardScaler: mean=0, std=1
# IMPORTANT: scale features only, NOT SalePrice
 
# Separate target from features
X = df.drop(columns=['SalePrice'])
y = df['SalePrice']   # already log-transformed in Cell 6
 
# Scale only numerical columns
num_cols = X.select_dtypes(include=['int64', 'float64']).columns
 
scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])
 
print(f"Features shape : {X.shape}")
print(f"Target shape   : {y.shape}")
 

Features shape : (2927, 188)
Target shape   : (2927,)


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size   = 0.2,      # 80% train, 20% test
    random_state= 42        # same split every time you run
)
 
print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")
 

X_train : (2341, 188)
X_test  : (586, 188)
y_train : (2341,)
y_test  : (586,)


In [10]:
train_df = X_train.copy()
train_df['SalePrice'] = y_train
 
test_df = X_test.copy()
test_df['SalePrice'] = y_test
 
train_df.to_csv(r'C:\git\machine learning\house-price-prediction\data\processed\train.csv', index=False)
test_df.to_csv(r'C:\git\machine learning\house-price-prediction\data\processed\test.csv',  index=False)
 
# Also save the scaler — you'll need it to inverse-transform predictions
import joblib
joblib.dump(scaler, r'C:\git\machine learning\house-price-prediction\data\processed\scaler.pkl')
 
print("Saved:")
print("  data/processed/train.csv")
print("  data/processed/test.csv")
print("  data/processed/scaler.pkl")
print(f"\nFinal dataset: {X_train.shape[1]} features, {len(X_train)} train rows, {len(X_test)} test rows")
print("\nPhase 3 complete ✓  →  Ready for Phase 4: Simple Linear Regression")

Saved:
  data/processed/train.csv
  data/processed/test.csv
  data/processed/scaler.pkl

Final dataset: 188 features, 2341 train rows, 586 test rows

Phase 3 complete ✓  →  Ready for Phase 4: Simple Linear Regression


In [11]:
df.head()

,MS SubClass,Lot Frontage,Lot Area,Land Slope,Overall Qual,Overall Cond,Year Built,Year Remod/Add,Mas Vnr Area,Exter Qual,...,Sale Type_ConLw,Sale Type_New,Sale Type_Oth,Sale Type_VWD,Sale Type_WD,Sale Condition_AdjLand,Sale Condition_Alloca,Sale Condition_Family,Sale Condition_Normal,Sale Condition_Partial
0,20,4.955827,10.366309,1,6,5,1960,1960,4.727388,3,...,False,False,False,False,True,False,False,False,True,False
1,20,4.394449,9.360741,1,5,6,1961,1961,0.000000,3,...,False,False,False,False,True,False,False,False,True,False
2,20,4.406719,9.565775,1,6,6,1958,1958,4.691348,3,...,False,False,False,False,True,False,False,False,True,False
3,20,4.543295,9.320181,1,7,5,1968,1968,0.000000,4,...,False,False,False,False,True,False,False,False,True,False
4,60,4.317488,9.534668,1,5,5,1997,1998,0.000000,3,...,False,False,False,False,True,False,False,False,True,False
